# System Design, Plainly — Part 3: Sharding (companion notebook)

One database can only get so big. Sharding splits it across many machines so storage
and write throughput scale with the number of shards. This notebook replays the four
simulations behind the post:

1. **Remap on scale-out** — `hash(key) % N` vs a consistent-hash ring when you add a node.
2. **The hot key** — a whale tenant that holds 12.3% of all rows, and why more shards don't help.
3. **Scatter-gather** — what a query without the shard key costs in throughput and p99.
4. **Range vs hash** — today's writes under time-range sharding vs user-hash sharding.

All numbers are toy simulations on synthetic data, labelled as such in the post.
Vendor figures quoted in the post (Slack 2.3M qps, GitHub 5.5M qps) are the companies'
own engineering-blog numbers, not reproduced here.


In [ ]:
import hashlib
import numpy as np

def h32(s):
    """Deterministic 32-bit hash (MD5 truncated) — stands in for Murmur/xxHash."""
    return int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16)

N_KEYS = 200_000
keys = np.arange(N_KEYS)
hashes = np.array([h32(k) for k in keys], dtype=np.int64)
print(f"{N_KEYS:,} keys hashed; sample: {hashes[:3]}")

In [ ]:
# --- 1. Remap on scale-out: hash(key) % N vs consistent-hash ring ---
def naive_moved(n0, n1):
    return float(np.mean((hashes % n0) != (hashes % n1)))

def ring_owner(n_nodes, vnodes_per_node=200):
    # vnode positions are FIXED per node (hash of node+vn id), so adding a node
    # only inserts new vnodes; existing vnodes never move.
    pos = np.array([h32(f"node-{n}-vn-{v}") for n in range(n_nodes)
                    for v in range(vnodes_per_node)], dtype=np.int64)
    node = np.repeat(np.arange(n_nodes), vnodes_per_node)
    order = np.argsort(pos)
    pos, node = pos[order], node[order]
    idx = np.searchsorted(pos, hashes % (2**32), side="left")
    idx = np.where(idx == len(pos), 0, idx)
    return node[idx]

own8 = ring_owner(8)
for n1 in (9, 10):
    print(f"8 -> {n1} shards: naive hash moves {naive_moved(8, n1)*100:.1f}% of keys, "
          f"ring moves {float(np.mean(own8 != ring_owner(n1)))*100:.1f}%")

# The textbook rule, verified: naive moves ~(n1-1)/n1 of keys when N grows by one
# step, the ring moves ~1/n1. Adding ONE node to an 8-node ring moves ~1/9 of keys.
assert abs(naive_moved(8, 9) - 8/9) < 0.01
assert abs(float(np.mean(own8 != ring_owner(9))) - 1/9) < 0.02

In [ ]:
# --- 2. The hot key: one whale tenant, indivisible across shards ---
# Whale tenant holds 12.3% of rows (the busiest Slack channel in the measured
# Vitess write-up); the other 87.7% spread over 999 tenants, Zipf-like.
n_tenants = 1000
whale = 0.123
rest = (1 - whale) * (1 / np.arange(2, n_tenants + 1)) ** 1.0
rest = rest / rest.sum() * (1 - whale)
shares = np.concatenate([[whale], rest])
rows = (shares * 1_000_000).astype(int)

def busiest_shard_share(n_shards):
    h = np.array([h32(f"tenant-{i}") for i in range(n_tenants)]) % n_shards
    per = np.zeros(n_shards)
    np.add.at(per, h, rows)
    return per.max() / rows.sum()

for n in (4, 8, 16, 32, 64, 128):
    print(f"{n:4d} shards: busiest holds {busiest_shard_share(n)*100:5.1f}% "
          f"(ideal: {100/n:5.2f}%)")
# The busiest shard asymptotes to the whale's 12.3% — it can never go below it.
assert abs(busiest_shard_share(128) - whale) < 0.01

In [ ]:
# --- 3. Scatter-gather: the shard key decides the price of a query ---
rng = np.random.default_rng(11)
n_sim, n_shards = 20000, 16
per_shard = rng.normal(8, 2, size=(n_sim, n_shards)).clip(min=1)  # ms per shard
targeted_p99 = float(np.percentile(rng.normal(8, 2, size=n_sim).clip(min=1), 99))
lookup_p99 = float(np.percentile(per_shard[:, :2].sum(axis=1), 99))   # lookup + fetch
scatter_p99 = float(np.percentile(per_shard.max(axis=1) + 6.0, 99))   # slowest + merge
print(f"p99 latency: targeted {targeted_p99:.1f} ms | lookup vindex {lookup_p99:.1f} ms "
      f"| scatter-gather {scatter_p99:.1f} ms")

# Throughput: each shard sustains 1,200 qps. A request consumes one shard-query
# per shard it touches, so cluster capacity = 16*1200 / (shard-queries per request).
per_shard_qps = 1200
for name, touched in [("targeted", 1), ("lookup vindex", 2), ("scatter-gather", 16)]:
    print(f"{name:15s}: {n_shards * per_shard_qps // touched:>6,} qps cluster-wide")
assert 16 * per_shard_qps // 16 == 1200  # scatter collapses 16x vs targeted

In [ ]:
# --- 4. Range vs hash: where today's writes land ---
# Range sharding by created_at: every write today lands on the shard that owns
# the newest range. Hash sharding by user_id: today's writes spread evenly.
shards = 8
range_today = np.zeros(shards); range_today[-1] = 100.0
hash_today = np.full(shards, 100.0 / shards)
print("today's writes, range-by-time (%):", range_today.astype(int))
print("today's writes, hash-by-user  (%):", hash_today.astype(int))
assert range_today.max() == 100.0 and hash_today.max() == 12.5

# Range scan of one week of data: range touches 1-2 shards, hash touches all 8.
print("1-week range scan touches: range = 1-2 shards, hash = all 8 shards")
print("Rule of thumb: hash spreads writes, range serves scans — pick by your query mix.")